# 1. Introducción: riesgo de entrega tardía en DataCo

Este Jupyter Book corresponde al **Primer Entregable** del proyecto de Machine
Learning. Utiliza *DataCo Smart Supply Chain for Big Data Analysis*
{cite}`constante_2019`, publicado en Mendeley Data y replicado en Kaggle, y
sigue la **ruta A (clasificación)** del enunciado **añadiendo las secciones 2.6 a 2.8**,
porque el dataset trae fecha y hora del pedido y coordenadas.

## Problema y pregunta de investigación

**Pregunta:** ¿Es posible predecir, **en el momento en que se registra un pedido**
y solo con información disponible en ese instante, si su entrega llegará **tarde**
respecto al plazo prometido?

Una buena predicción permite priorizar pedidos (reasignar transporte, avisar al
cliente, reforzar el despacho) **antes** de que el retraso ocurra. Por eso el
momento de predicción es la creación del pedido, y cualquier variable que se
conozca después (días reales de envío, estado de la entrega, estado del pedido,
fecha de despacho) es **fuga de datos** y se excluye (capítulo 9). Este libro
trabaja siempre con la versión **«DataCo (sin fuga)»**.

## Variable objetivo

`Late_delivery_risk` (renombrada `late_delivery_risk`):

- `1` → la entrega fue tardía (los días reales superan los programados);
- `0` → la entrega fue adelantada, a tiempo o el pedido fue cancelado.

## Métrica principal: ROC-AUC

La métrica principal es el **área bajo la curva ROC (ROC-AUC)**:

- el uso previsto es **ordenar** pedidos por riesgo para priorizarlos, y la ROC-AUC mide
  exactamente la calidad de ese orden (probabilidad de que un pedido tardío reciba
  un puntaje mayor que uno puntual);
- no depende del umbral, que la operación puede ajustar después según su capacidad;
- las clases están **moderadamente balanceadas** (capítulo 3), de modo que la ROC-AUC
  no queda inflada por una clase mayoritaria; aun así se reportan accuracy,
  recall, F1, Average Precision, Brier, matriz de confusión y calibración.

In [1]:
from pathlib import Path
import re, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

project_root = Path.cwd().resolve()
while not (project_root / "_config.yml").exists():
    if project_root.parent == project_root:
        raise FileNotFoundError("No se encontró _config.yml. Ejecuta el notebook dentro de la carpeta del libro.")
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

for _folder in ("tables", "figures"):          # se crean si no existen
    (project_root / _folder).mkdir(exist_ok=True)

from src.project_utils import *

configure_plots()
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 70)
TABLES = project_root / "tables"

def md_out(text):
    display(Markdown(text))

## Fuente, licencia y alcance del dataset

- **Autores:** Fabian Constante, Fernando Silva y António Pereira (2019).
- **Fuente original:** Mendeley Data, DOI [`10.17632/8gx2fvg2k6.5`](https://data.mendeley.com/datasets/8gx2fvg2k6/5).
- **Réplica usada:** [Kaggle — DataCo Smart Supply Chain for Big Data Analysis](https://www.kaggle.com/datasets/shashwatwork/dataco-smart-supply-chain-for-big-data-analysis).
- **Licencia:** CC BY 4.0 (Mendeley Data); uso académico con atribución.
- **Archivo:** `DataCoSupplyChainDataset.csv` (codificación latin-1).

La empresa (DataCo Global) es ficticia y vende artículos deportivos, ropa y
electrónica; los registros simulan su cadena de suministro entre 2015 y 2018. La
**unidad de observación es una línea (ítem) de un pedido**; un pedido contiene
de 1 a 5 líneas y un cliente realiza varios pedidos.

El libro busca primero `data/DataCoSupplyChainDataset.csv`; si no está, lo
extrae del ZIP incluido en `data/`; si tampoco existe, intenta descargar una
copia pública y, si falla, indica exactamente qué archivo copiar. No se usan
rutas absolutas.

In [2]:
raw = load_raw()
n_rows, n_cols = raw.shape
n_orig = len(RENAME)
md_out(f"- **Filas (líneas de pedido):** {es(n_rows, 0)} (esperadas según la ficha: {es(EXPECTED_ROWS, 0)})\n"
       f"- **Columnas originales:** {n_orig}; tras derivar variables de calendario: {n_cols}\n"
       f"- **Pedidos distintos:** {es(raw[ORDER_COL].nunique(), 0)}; **clientes distintos:** {es(raw[CUSTOMER_COL].nunique(), 0)}\n"
       f"- **Periodo:** {raw[DATE_COL].min():%Y-%m-%d} a {raw[DATE_COL].max():%Y-%m-%d}\n"
       f"- **Mínimo del enunciado:** 20.000 observaciones → {'cumple' if n_rows >= 20000 else 'NO cumple'}")
assert n_rows == EXPECTED_ROWS, "El archivo no tiene el número de filas documentado"
muestra = [c for c in RENAME.values() if c not in PII_COLS][:14]
raw[muestra].head(3).T

- **Filas (líneas de pedido):** 180.519 (esperadas según la ficha: 180.519)
- **Columnas originales:** 53; tras derivar variables de calendario: 68
- **Pedidos distintos:** 65.752; **clientes distintos:** 20.652
- **Periodo:** 2015-01-01 a 2018-01-31
- **Mínimo del enunciado:** 20.000 observaciones → cumple

,0,1,2
payment_type,CASH,PAYMENT,PAYMENT
days_shipping_real,2,3,3
days_shipping_scheduled,4,4,4
benefit_per_order,88.790001,91.18,68.25
sales_per_customer,239.979996,193.990005,227.5
delivery_status,Advance shipping,Advance shipping,Advance shipping
late_delivery_risk,0,0,0
category_id,43,48,24
category_name,Camping & Hiking,Water Sports,Women's Apparel
customer_city,Hickory,Chicago,Chicago


## Elección de la ruta del enunciado

El enunciado distingue cuatro rutas según el tipo de dataset y pide añadir las
secciones 2.6 a 2.8 cuando una clasificación tiene tiempo o espacio. Se comprueba
buscando esas columnas.

In [3]:
fechas = [ORIGINAL_NAME[c] for c in RENAME.values() if re.search(r"date", c)]
coords = [ORIGINAL_NAME[c] for c in (LAT, LON)]
md_out(f"- Objetivo: `{ORIGINAL_NAME[TARGET]}` con valores {sorted(int(v) for v in raw[TARGET].unique())} → **clasificación binaria (ruta A)**.\n"
       f"- Columnas con fecha u hora: **{fechas}**.\n"
       f"- Columnas con coordenadas: **{coords}**.\n\n"
       "**Conclusión:** se sigue la **ruta A** y se añaden las secciones **2.6 (temporal)**, **2.7 (espacial)** y **2.8 (espacio-temporal)** "
       "(capítulos 7 y 8). La fecha del pedido define el momento de predicción; la fecha de despacho es posterior y se trata como fuga.")

- Objetivo: `Late_delivery_risk` con valores [0, 1] → **clasificación binaria (ruta A)**.
- Columnas con fecha u hora: **['order date (DateOrders)', 'shipping date (DateOrders)']**.
- Columnas con coordenadas: **['Latitude', 'Longitude']**.

**Conclusión:** se sigue la **ruta A** y se añaden las secciones **2.6 (temporal)**, **2.7 (espacial)** y **2.8 (espacio-temporal)** (capítulos 7 y 8). La fecha del pedido define el momento de predicción; la fecha de despacho es posterior y se trata como fuga.

## Partición entrenamiento / prueba

El enunciado exige **reservar la prueba antes de cualquier decisión** y que la
partición respete la estructura: **cronológica** si hay tiempo y **por grupos** si
hay varias filas por entidad.

- **Cronológica:** el modelo se usará sobre pedidos futuros; se entrena con el
  pasado y se evalúa con el **último 20 % del periodo** (en filas).
- **Separación (*gap*) de 7 días:** la etiqueta de un pedido solo se conoce
  cuando se entrega (hasta 6 días después). Se descartan los pedidos de la semana
  anterior al corte, porque en operación su resultado aún no se conocería.
- **Por pedido:** las líneas de un pedido comparten fecha y etiqueta; al cortar
  por fecha ningún pedido queda dividido (se verifica abajo).

La validación cruzada del capítulo 11 usa el mismo esquema (ventana creciente
con *gap* de 7 días). La partición queda guardada en `tables/particion_pedidos.csv`.

In [4]:
train_all, test_all = load_train(), load_test()
gap_all = load_gap()
gap_start, cutoff = split_boundaries()
shared_orders = set(train_all[ORDER_COL]) & set(test_all[ORDER_COL])
assert len(shared_orders) == 0, "Hay pedidos en entrenamiento y prueba a la vez"
assert train_all[DATE_COL].max() < gap_start <= cutoff <= test_all[DATE_COL].min()

resumen = pd.DataFrame({
    "conjunto": ["Entrenamiento", "Gap (descartado)", "Prueba"],
    "desde": [train_all[DATE_COL].min(), gap_all[DATE_COL].min(), test_all[DATE_COL].min()],
    "hasta": [train_all[DATE_COL].max(), gap_all[DATE_COL].max(), test_all[DATE_COL].max()],
    "líneas": [len(train_all), len(gap_all), len(test_all)],
    "pedidos": [train_all[ORDER_COL].nunique(), gap_all[ORDER_COL].nunique(), test_all[ORDER_COL].nunique()],
    "entrega_tardía_%": [100 * train_all[TARGET].mean(), 100 * gap_all[TARGET].mean(), 100 * test_all[TARGET].mean()],
})
display(resumen.round(2))
clientes_comp = len(set(train_all[CUSTOMER_COL]) & set(test_all[CUSTOMER_COL]))
md_out(f"Corte: pedidos anteriores a **{gap_start:%Y-%m-%d}** para entrenamiento y desde **{cutoff:%Y-%m-%d}** para prueba. "
       f"Pedidos en ambos conjuntos: **{len(shared_orders)}**. Clientes con pedidos en ambos: {es(clientes_comp, 0)} "
       "(es lo esperado en operación: los clientes vuelven; `customer_id` nunca se usa como predictor). "
       f"La tasa de entregas tardías es {es(train_all[TARGET].mean(), 1, pct=True)} en entrenamiento y {es(test_all[TARGET].mean(), 1, pct=True)} en prueba. "
       "A partir de aquí, **todo el EDA y todas las decisiones usan solo el entrenamiento**; la prueba se usa de nuevo en el capítulo 11.")

,conjunto,desde,hasta,líneas,pedidos,entrega_tardía_%
0,Entrenamiento,2015-01-01 00:00:00,2017-04-14 23:42:00,143075,47719,54.90
1,Gap (descartado),2017-04-15 00:03:00,2017-04-21 23:52:00,1222,403,50.00
2,Prueba,2017-04-22 00:13:00,2018-01-31 23:38:00,36222,17630,54.71


Corte: pedidos anteriores a **2017-04-15** para entrenamiento y desde **2017-04-22** para prueba. Pedidos en ambos conjuntos: **0**. Clientes con pedidos en ambos: 6.426 (es lo esperado en operación: los clientes vuelven; `customer_id` nunca se usa como predictor). La tasa de entregas tardías es 54,9 % en entrenamiento y 54,7 % en prueba. A partir de aquí, **todo el EDA y todas las decisiones usan solo el entrenamiento**; la prueba se usa de nuevo en el capítulo 11.

## Mapa de capítulos

| Capítulo | Contenido | Sección del enunciado |
|---|---|---|
| 2 | Base de datos: estructura, tamaño, calidad, sesgos, ética | Base de datos (10 %) |
| 3 | Variable objetivo | 2.1 |
| 4 | Análisis univariado | 2.2 |
| 5 | Análisis bivariado | 2.3 |
| 6 | Análisis multivariado | 2.4 |
| 7 | Componente temporal | 2.6 |
| 8 | Componente espacial y espacio-temporal | 2.7 y 2.8 |
| 9 | Auditoría de fuga de datos (DataCo sin fuga) | 2.5 |
| 10 | Preprocesamiento y Pipeline | 2.9 |
| 11 | Modelo base (Dummy y regresión logística, ROC-AUC) | Modelo base (30 %) |
| 12 | Conclusiones y checklist | — |

## Consideraciones éticas (resumen)

El archivo contiene nombres, direcciones y códigos postales de clientes
(correo y contraseña vienen enmascarados). Aunque la empresa es ficticia, se
tratan como datos personales: **no se usan en el modelo ni se muestran** en el
libro. El capítulo 2 discute el riesgo de reidentificación y el capítulo 12 los
usos apropiados.